# Antlings Internship — AI/ML Technical Assessment
## Drone Human & Car Detection + Counting System

**Dataset:** VisDrone2019-DET | **Model:** YOLOv8n | **Tracker:** ByteTrack | **Hardware:** AMD MI300X (ROCm 7.0)

| Task | Description | Weight |
|------|-------------|--------|
| Task 01 | Dataset Understanding & Preprocessing | 20% |
| Task 02 | Model Training | 30% |
| Task 03 | Human & Car Detection with Counting | 20% |
| Task 04 | Object Tracking — ByteTrack *(Bonus)* | — |
| Task 05 | Evaluation & Visualization | 15% |


## Setup — Environment & Dependencies

In [ ]:
import subprocess, os, glob, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import torch

def run(cmd): subprocess.run(cmd, shell=True, check=True)

print(f'PyTorch  : {torch.__version__}')
print(f'GPU      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


PyTorch  : 2.9.0.dev20250821+rocm7.0.0.git125803b7
GPU      : True
Device   : 
VRAM     : 205.8 GB


In [ ]:
run('pip install -q ultralytics kagglehub')
from ultralytics import YOLO
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')


## Task 01 — Dataset Understanding & Preprocessing

### 1.1 Download Dataset

**VisDrone2019-DET** is a large-scale drone imagery benchmark from Tianjin University.  
Downloaded via `kagglehub` directly onto the cloud server — nothing touches the local machine.


In [ ]:
import kagglehub

path = kagglehub.dataset_download('banuprasadb/visdrone-dataset')
DATASET_DIR = f'{path}/VisDrone_Dataset'
print(f'Dataset location: {DATASET_DIR}')


Dataset location: /root/.cache/kagglehub/datasets/banuprasadb/visdrone-dataset/versions/1/VisDrone_Dataset


### 1.2 Dataset Structure

The Kaggle version is **pre-converted to YOLO format** — labels are normalized bounding boxes, no manual conversion needed.

**10 Classes:**
| ID | Class | Our mapping |
|----|-------|-------------|
| 0 | pedestrian | → **human** |
| 1 | people | → **human** |
| 2 | bicycle | ignored |
| 3 | car | → **car** |
| 4–9 | van, truck, tricycle, bus, motor… | ignored |

We merge `pedestrian` and `people` into one **human** class — both represent people, just at different visibility levels.


In [ ]:
VISDRONE_CLASSES = {
    0:'pedestrian', 1:'people', 2:'bicycle', 3:'car',
    4:'van', 5:'truck', 6:'tricycle', 7:'awning-tricycle',
    8:'bus', 9:'motor'
}
HUMAN_IDS = [0, 1]
CAR_IDS   = [3]

for split in ['VisDrone2019-DET-train','VisDrone2019-DET-val','VisDrone2019-DET-test-dev']:
    imgs = len(glob.glob(f'{DATASET_DIR}/{split}/images/*.jpg'))
    lbls = len(glob.glob(f'{DATASET_DIR}/{split}/labels/*.txt'))
    print(f'{split}: {imgs} images, {lbls} labels')


VisDrone2019-DET-train: 6471 images, 6471 labels
VisDrone2019-DET-val: 548 images, 548 labels
VisDrone2019-DET-test-dev: 1610 images, 1610 labels


### 1.3 Class Distribution & Statistics

In [ ]:
label_files = sorted(glob.glob(f'{DATASET_DIR}/VisDrone2019-DET-train/labels/*.txt'))
class_counts = {i:0 for i in range(10)}
box_sizes = []

for lf in tqdm(label_files[:1000], desc='Analyzing labels'):
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            w, h = float(parts[3]), float(parts[4])
            class_counts[cid] = class_counts.get(cid,0) + 1
            box_sizes.append((cid, w, h, w*h))

df_boxes = pd.DataFrame(box_sizes, columns=['class_id','w','h','area'])

print('Class distribution (1000-image sample):')
for cid, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    bar = '█' * (cnt//50)
    print(f'  {cid} {VISDRONE_CLASSES[cid]:20s}: {cnt:5d}  {bar}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14,5))
names = [VISDRONE_CLASSES[i] for i in range(10)]
counts = [class_counts[i] for i in range(10)]
colors = ['#22C55E' if i in HUMAN_IDS else '#EF4444' if i in CAR_IDS else '#64748B' for i in range(10)]
axes[0].bar(names, counts, color=colors, edgecolor='white')
axes[0].set_title('All Classes — VisDrone Train Set', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Count')

target = df_boxes[df_boxes['class_id'].isin(HUMAN_IDS + CAR_IDS)].copy()
target['name'] = target['class_id'].map(VISDRONE_CLASSES)
for name, grp in target.groupby('name'):
    axes[1].hist(grp['area'].clip(upper=0.005), bins=50, alpha=0.7, label=name)
axes[1].set_title('Bounding Box Area Distribution (Target Classes)', fontsize=12)
axes[1].set_xlabel('Normalized Area')
axes[1].legend()
plt.tight_layout()
plt.savefig('task01_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


### 1.4 Sample Visualizations

Ground truth bounding boxes from training data. **Green = human, Red = car, Gray = other.**


In [ ]:
def visualize_gt(img_path, lbl_path, ax):
    img = np.array(Image.open(img_path).convert('RGB'))
    H, W = img.shape[:2]
    ax.imshow(img)
    h_count = c_count = 0
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) < 5: continue
                cid = int(p[0])
                xc,yc,w,h = float(p[1]),float(p[2]),float(p[3]),float(p[4])
                x1,y1 = int((xc-w/2)*W), int((yc-h/2)*H)
                color = '#22C55E' if cid in HUMAN_IDS else '#EF4444' if cid in CAR_IDS else '#94A3B8'
                rect = patches.Rectangle((x1,y1),int(w*W),int(h*H),lw=1,edgecolor=color,facecolor='none')
                ax.add_patch(rect)
                if cid in HUMAN_IDS: h_count+=1
                elif cid in CAR_IDS: c_count+=1
    ax.set_title(f'Humans:{h_count} Cars:{c_count}', fontsize=9)
    ax.axis('off')

train_imgs = sorted(glob.glob(f'{DATASET_DIR}/VisDrone2019-DET-train/images/*.jpg'))
random.seed(42)
samples = random.sample(train_imgs, 6)

fig, axes = plt.subplots(2, 3, figsize=(16,9))
for ax, img_path in zip(axes.flatten(), samples):
    lbl_path = img_path.replace('images','labels').replace('.jpg','.txt')
    visualize_gt(img_path, lbl_path, ax)
plt.suptitle('Task 01 — Ground Truth Annotations (VisDrone Training Set)', fontsize=13)
plt.tight_layout()
plt.savefig('task01_samples.png', dpi=120, bbox_inches='tight')
plt.show()


### 1.5 Dataset Challenges

| Challenge | Description | Solution Applied |
|-----------|-------------|------------------|
| **Small objects** | Humans appear as 10–30px blobs from altitude. >60% of annotations have area <0.1% of image | `imgsz=640`, mosaic augmentation |
| **High density** | 100+ humans per frame. NMS suppresses valid detections as duplicates | Lower NMS IoU threshold (`iou=0.45`) |
| **Class imbalance** | Pedestrian class dominates heavily | Mosaic + copy-paste augmentation |
| **Scale variation** | Same object at vastly different sizes per drone altitude | `scale=0.5` augmentation |
| **Viewpoint diversity** | Top-down and oblique angles both present | `flipud=0.3` augmentation |
| **Occlusion** | Heavy overlap in crowd scenes | Lower confidence threshold |


### 1.6 Dataset Configuration (data.yaml)

In [ ]:
# The pre-existing yaml uses relative paths — we rewrite with absolute paths
# so YOLOv8 can find the data regardless of working directory

yaml_content = f'''path: {DATASET_DIR}
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images

nc: 10
names:
  0: pedestrian\n  1: people\n  2: bicycle\n  3: car
  4: van\n  5: truck\n  6: tricycle\n  7: awning-tricycle
  8: bus\n  9: motor'''

yaml_path = f'{DATASET_DIR}/data.yaml'
with open(yaml_path, 'w') as f: f.write(yaml_content)
print('data.yaml written to:', yaml_path)


## Task 02 — Model Training

### 2.1 Model Selection

**YOLOv8n (nano)** fine-tuned from COCO pretrained weights.

- **Why YOLOv8?** Single-stage detector — fast, accurate, built-in VisDrone augmentation support and ByteTrack tracking
- **Why fine-tune, not train from scratch?** COCO pretrained weights already know edges, shapes, objects. We adapt this knowledge to aerial views
- **Why train on all 10 classes?** Richer feature learning — better representations benefit human/car detection. We filter at inference only

### 2.2 Augmentation Strategy

| Parameter | Value | Reason |
|-----------|-------|--------|
| `mosaic` | 1.0 | Combines 4 images — increases density & scale variety per batch |
| `copy_paste` | 0.1 | Copies objects across images — counters class imbalance |
| `flipud` | 0.3 | Vertical flip valid for aerial (not ground cameras) |
| `degrees` | 10° | Covers angled drone captures |
| `scale` | 0.5 | Simulates different flight altitudes |

**Hardware:** AMD MI300X GPU (192GB VRAM) via AMD Developer Cloud, ROCm 7.0


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # COCO pretrained starting point
print(f'Training on: {"GPU (MI300X)" if torch.cuda.is_available() else "CPU"}')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    patience=15,          # early stopping
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=1.0,
    copy_paste=0.1,
    flipud=0.3,
    fliplr=0.5,
    degrees=10.0,
    scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    name='visdrone_detection',
    project='runs',
    exist_ok=True,
)

BEST_WEIGHTS = f'{results.save_dir}/weights/best.pt'
print(f'Best weights: {BEST_WEIGHTS}')


### 2.3 Training Curves

In [ ]:
from IPython.display import Image as IPImage, display

curves = f'{results.save_dir}/results.png'
if os.path.exists(curves):
    display(IPImage(filename=curves, width=900))


## Task 03 — Human & Car Detection with Counting

### 3.1 Detection Pipeline

```
Image → YOLOv8 inference → filter classes [0,1,3] → count humans → draw boxes + overlay
```

- **Human** = class 0 (pedestrian) + class 1 (people) — merged for complete count
- **Car** = class 3
- **conf=0.25** — detections below 25% confidence discarded
- **iou=0.45** — NMS threshold; lower keeps more overlapping boxes (important for dense crowds)


In [ ]:
BEST_WEIGHTS = 'runs/detect/visdrone_detection/weights/best.pt'
det_model = YOLO(BEST_WEIGHTS)
HUMAN_IDS = [0, 1]
CAR_IDS   = [3]

def detect_and_count(image_path, conf=0.25, iou=0.45, save_path=None):
    results = det_model.predict(source=image_path, conf=conf, iou=iou, verbose=False)[0]
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    human_count = car_count = 0

    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf_score = float(box.conf[0])
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        if cls_id in HUMAN_IDS:
            human_count += 1; color = (34,197,94); label = f'H {conf_score:.2f}'
        elif cls_id in CAR_IDS:
            car_count += 1; color = (239,68,68); label = f'C {conf_score:.2f}'
        else: continue
        cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
        (lw,lh),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.4,1)
        cv2.rectangle(img,(x1,y1-lh-4),(x1+lw+2,y1),color,-1)
        cv2.putText(img,label,(x1+1,y1-2),cv2.FONT_HERSHEY_SIMPLEX,0.4,(255,255,255),1)

    # Count overlay
    cv2.rectangle(img,(8,8),(270,78),(10,15,30),-1)
    cv2.putText(img,f'Humans : {human_count}',(16,34),cv2.FONT_HERSHEY_SIMPLEX,0.7,(34,197,94),2)
    cv2.putText(img,f'Cars   : {car_count}',(16,68),cv2.FONT_HERSHEY_SIMPLEX,0.7,(239,68,68),2)
    if save_path: cv2.imwrite(save_path, cv2.cvtColor(img,cv2.COLOR_RGB2BGR))
    return img, human_count, car_count

print('detect_and_count() ready')


### 3.2 Detection Results

In [ ]:
test_imgs = sorted(glob.glob(f'{DATASET_DIR}/VisDrone2019-DET-test-dev/images/*.jpg'))
random.seed(7)
sample_test = random.sample(test_imgs, 6)

fig, axes = plt.subplots(2,3,figsize=(18,10))
fig.patch.set_facecolor('#0F172A')
for i,(ax,img_path) in enumerate(zip(axes.flatten(),sample_test)):
    annotated, h, c = detect_and_count(img_path, save_path=f'task03_result_{i+1}.jpg')
    ax.imshow(annotated)
    ax.set_title(f'Humans: {h}  |  Cars: {c}', color='white', fontsize=11)
    ax.axis('off')
plt.suptitle('Task 03 — Human & Car Detection with Counting', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('task03_detection_grid.png', dpi=150, bbox_inches='tight', facecolor='#0F172A')
plt.show()


### 3.3 Batch Inference & Count CSV

In [ ]:
os.makedirs('task03_batch_output', exist_ok=True)
records = []
for img_path in tqdm(test_imgs[:50], desc='Batch inference'):
    fname = os.path.basename(img_path)
    _, h, c = detect_and_count(img_path, save_path=f'task03_batch_output/{fname}')
    records.append({'image':fname,'humans':h,'cars':c})

df = pd.DataFrame(records)
df.to_csv('task03_batch_output/counts.csv', index=False)
print(f'Processed {len(records)} images')
print(df[['humans','cars']].describe().round(1))


## Task 04 — Object Tracking with ByteTrack *(Bonus)*

### Why Tracking?
Detection answers *where are objects in this frame?*  
Tracking answers *which object in frame 1 is the same as frame 2?* — giving persistent IDs and trajectory trails.

### ByteTrack Algorithm
1. Associates **all** detections (not just high-confidence) with existing tracks using IoU
2. Kalman filter predicts next position based on object velocity
3. More robust than DeepSORT in dense scenes — no re-ID model needed

> **Note:** The tracking video was generated from a sequence of unrelated test images rather than
> a true drone video feed. ByteTrack requires temporal continuity — in real drone footage,
> tracking would be significantly more stable with consistent track IDs across all frames.


In [ ]:
# Create image sequence video using ffmpeg
test_imgs_all = sorted(glob.glob(f'{DATASET_DIR}/VisDrone2019-DET-test-dev/images/*.jpg'))
os.makedirs('temp_frames', exist_ok=True)
import shutil
for i, img_path in enumerate(test_imgs_all[:60]):
    shutil.copy2(img_path, f'temp_frames/frame_{i:04d}.jpg')

os.system('ffmpeg -y -framerate 5 -i temp_frames/frame_%04d.jpg -c:v libx264 -pix_fmt yuv420p test_sequence.mp4')
VIDEO_PATH = 'test_sequence.mp4'
print(f'Video: {os.path.getsize(VIDEO_PATH)/1e6:.1f} MB' if os.path.exists(VIDEO_PATH) else 'Failed')


In [ ]:
track_model = YOLO(BEST_WEIGHTS)

def track_video(video_path, output_path='task04_tracked.avi', conf=0.25):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 5
    W,H = int(cap.get(3)), int(cap.get(4))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'XVID'), fps, (W,H))
    track_history, frame_idx = {}, 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        results = track_model.track(frame, conf=conf, persist=True,
                                    classes=HUMAN_IDS+CAR_IDS, verbose=False)[0]
        h_count = c_count = 0
        if results.boxes.id is not None:
            for box,tid,cid in zip(results.boxes.xyxy,
                                    results.boxes.id.int(),
                                    results.boxes.cls.int()):
                x1,y1,x2,y2 = map(int,box)
                tid,cid = int(tid),int(cid)
                color = (34,197,94) if cid in HUMAN_IDS else (239,68,68)
                prefix = 'H' if cid in HUMAN_IDS else 'C'
                if cid in HUMAN_IDS: h_count+=1
                else: c_count+=1
                cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
                cv2.putText(frame,f'{prefix}#{tid}',(x1,max(y1-4,10)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.4,color,1)
                cx,cy = (x1+x2)//2,(y1+y2)//2
                track_history.setdefault(tid,[]).append((cx,cy))
                track_history[tid] = track_history[tid][-20:]
                pts = track_history[tid]
                for j in range(1,len(pts)): cv2.line(frame,pts[j-1],pts[j],color,1)
        cv2.rectangle(frame,(8,8),(270,78),(10,15,30),-1)
        cv2.putText(frame,f'Humans : {h_count}',(16,34),cv2.FONT_HERSHEY_SIMPLEX,0.7,(34,197,94),2)
        cv2.putText(frame,f'Cars   : {c_count}',(16,68),cv2.FONT_HERSHEY_SIMPLEX,0.7,(239,68,68),2)
        out.write(frame)
        frame_idx+=1
    cap.release(); out.release()
    print(f'Tracking complete: {output_path} ({frame_idx} frames)')

track_video(VIDEO_PATH)


## Task 05 — Evaluation & Visualization

### 5.1 Quantitative Metrics

- **Precision** — of all detections, what % were correct? High precision = few false alarms
- **Recall** — of all real objects, what % did we find? High recall = few misses
- **mAP@0.5** — Mean Average Precision at 50% IoU. Primary benchmark metric
- **mAP@50:95** — Stricter: averaged across IoU thresholds 0.5 to 0.95


In [ ]:
eval_model = YOLO(BEST_WEIGHTS)
metrics = eval_model.val(data=yaml_path, split='val', conf=0.25, iou=0.5, verbose=False)

print('='*50)
print('EVALUATION RESULTS — VisDrone Val Set (548 images)')
print('='*50)
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')
print()
VISDRONE_CLASSES = {0:'pedestrian',1:'people',2:'bicycle',3:'car',4:'van',
                    5:'truck',6:'tricycle',7:'awning-tricycle',8:'bus',9:'motor'}
print('Per-class mAP@0.5:')
for i,name in VISDRONE_CLASSES.items():
    if i < len(metrics.box.ap50):
        print(f'  {name:20s}: {metrics.box.ap50[i]:.4f}')


EVALUATION RESULTS — VisDrone Val Set (548 images)
mAP@0.5      : 0.2026
mAP@0.5:0.95 : 0.1212
Precision    : 0.5356
Recall       : 0.2489


### 5.2 Inference Speed

In [ ]:
speed_imgs = test_imgs[:30]
start = time.time()
for p in speed_imgs: det_model.predict(p, conf=0.25, verbose=False)
elapsed = time.time()-start
fps = len(speed_imgs)/elapsed
print(f'FPS: {fps:.1f} | ms/frame: {elapsed/len(speed_imgs)*1000:.1f}ms')


### 5.3 Metrics Dashboard

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.patch.set_facecolor('#0F172A')

names3 = ['pedestrian','people','car']
map3   = [metrics.box.ap50[i] for i in [0,1,3]]
colors3= ['#22C55E','#86EFAC','#EF4444']
bars = axes[0].bar(names3,map3,color=colors3,edgecolor='#334155',width=0.5)
axes[0].set_ylim(0,1)
axes[0].set_facecolor('#1E293B')
axes[0].set_title('mAP@0.5 — Target Classes',color='white',fontsize=12)
axes[0].tick_params(colors='white')
for bar,val in zip(bars,map3):
    axes[0].text(bar.get_x()+bar.get_width()/2,val+0.02,f'{val:.3f}',
                 ha='center',color='white',fontsize=11,fontweight='bold')

mnames = ['Precision','Recall','mAP@0.5','mAP@50:95']
mvals  = [metrics.box.mp,metrics.box.mr,metrics.box.map50,metrics.box.map]
bars2  = axes[1].barh(mnames,mvals,color=['#3B82F6','#8B5CF6','#F59E0B','#EC4899'],edgecolor='#334155')
axes[1].set_xlim(0,1)
axes[1].set_facecolor('#1E293B')
axes[1].set_title('Overall Detection Metrics',color='white',fontsize=12)
axes[1].tick_params(colors='white')
for bar,val in zip(bars2,mvals):
    axes[1].text(val+0.01,bar.get_y()+bar.get_height()/2,f'{val:.3f}',
                 va='center',color='white',fontsize=11)

plt.suptitle('Task 05 — Evaluation Results',color='white',fontsize=14)
plt.tight_layout()
plt.savefig('task05_metrics.png',dpi=150,bbox_inches='tight',facecolor='#0F172A')
plt.show()


### 5.4 Strengths, Limitations & Future Work

**Strengths**
- YOLOv8 on AMD MI300X: ~2.3ms/image — real-time capable for drone deployment
- Training on all 10 classes → richer feature learning benefits target class detection
- Mosaic + copy-paste augmentation specifically addresses VisDrone's small/dense object challenges
- ByteTrack provides persistent IDs with no extra model overhead
- Dual-class human merging (pedestrian + people) gives more complete counts

**Limitations**
- YOLOv8n is the smallest variant — yolov8m/l would push mAP significantly higher
- 50 epochs is a baseline — SOTA VisDrone results require 300+ epochs
- Frame-by-frame counting doesn't track unique persons across video frames
- Tracking video generated from unrelated test images — real drone footage would show more stable track IDs

**Future Improvements**
1. **SAHI** (Slicing Aided Hyper Inference) — tile images, detect on tiles, merge results → dramatically improves small object recall
2. **Higher resolution** — `imgsz=1280` for finer detail
3. **Track-ID-based unique counting** — count unique track IDs per video segment for true crowd count
4. **Larger model** — YOLOv8m/l or RT-DETR for higher mAP
